# Feedback Loops in Instacart Recommendations

Reproducible notebook for the paper results.
Run cells top-to-bottom once, then re-run individual result cells as needed.

## 0) Setup

This notebook assumes you run from repo root and raw CSVs are present there.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

In [2]:
def resolve_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur] + list(cur.parents):
        if (candidate / "orders.csv").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find orders.csv by searching upward from {start.resolve()}"
    )

ROOT = resolve_repo_root(Path.cwd())
OUT_DIR = ROOT / "analysis" / "final_presentation_analysis_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT, OUT_DIR


(PosixPath('/Users/cameronwang/Desktop/Foundations of Algorithmic Unfairness/unfairness-instacart-project'),
 PosixPath('/Users/cameronwang/Desktop/Foundations of Algorithmic Unfairness/unfairness-instacart-project/analysis/final_presentation_analysis_outputs'))

## 1) Data (Section 3.1)

Load prior-order data and confirm sample sizes.

In [3]:
orders = pd.read_csv(
    ROOT / "orders.csv",
    usecols=["order_id", "user_id", "eval_set", "order_number"],
    dtype={"order_id": "int32", "user_id": "int32", "eval_set": "category", "order_number": "int16"},
)
orders_prior = orders.loc[orders["eval_set"] == "prior", ["order_id", "user_id", "order_number"]].copy()

op_prior = pd.read_csv(
    ROOT / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    dtype={"order_id": "int32", "product_id": "int32"},
)

products = pd.read_csv(
    ROOT / "products.csv",
    usecols=["product_id", "aisle_id"],
    dtype={"product_id": "int32", "aisle_id": "int16"},
)

print("n_users:", orders_prior["user_id"].nunique())
print("n_prior_orders:", len(orders_prior))

n_users: 206209
n_prior_orders: 3214874


## 2) Popularity Segmentation (Section 3.2)

In [4]:
product_counts = op_prior["product_id"].value_counts().sort_index()
q90 = float(product_counts.quantile(0.90))
q99 = float(product_counts.quantile(0.99))

popularity = pd.DataFrame({
    "product_id": product_counts.index.astype(np.int32),
    "global_count": product_counts.values.astype(np.int32),
})
popularity["is_popular"] = (popularity["global_count"] >= q90).astype(np.int8)
popularity["is_ultra_popular"] = (popularity["global_count"] >= q99).astype(np.int8)
popularity["log_global_count"] = np.log(popularity["global_count"].astype(float))

print("popular threshold q90:", q90)
print("ultra-popular threshold q99:", q99)

popular threshold q90: 1021.0
ultra-popular threshold q99: 9931.159999999982


## 3) Build Order-Level Metrics (Section 3.3)

In [5]:
line_items = op_prior.merge(popularity, on="product_id", how="left", validate="many_to_one")

order_metrics = (
    line_items.groupby("order_id", sort=False)
    .agg(
        avg_popular_share=("is_popular", "mean"),
        avg_ultra_popular_share=("is_ultra_popular", "mean"),
        avg_mean_log_popularity=("log_global_count", "mean"),
        avg_items=("product_id", "size"),
        n_unique_items=("product_id", "nunique"),
    )
    .reset_index()
)
order_metrics["avg_unique_ratio"] = order_metrics["n_unique_items"] / order_metrics["avg_items"]
order_metrics.drop(columns=["n_unique_items"], inplace=True)

# HHI across aisle-share concentration within each order
line_items_aisle = op_prior.merge(products, on="product_id", how="left", validate="many_to_one")
aisle_counts = (
    line_items_aisle.groupby(["order_id", "aisle_id"], sort=False)
    .size()
    .rename("aisle_item_count")
    .reset_index()
)
order_totals = aisle_counts.groupby("order_id", sort=False)["aisle_item_count"].sum().rename("order_items")
aisle_counts = aisle_counts.merge(order_totals, on="order_id", how="left", validate="many_to_one")
aisle_counts["aisle_share_sq"] = (aisle_counts["aisle_item_count"] / aisle_counts["order_items"]) ** 2
order_hhi = aisle_counts.groupby("order_id", sort=False)["aisle_share_sq"].sum().rename("avg_hhi").reset_index()

order_metrics = order_metrics.merge(orders_prior, on="order_id", how="left", validate="one_to_one")
order_metrics = order_metrics.merge(order_hhi, on="order_id", how="left", validate="one_to_one")
order_metrics.head()

,order_id,avg_popular_share,avg_ultra_popular_share,avg_mean_log_popularity,avg_items,avg_unique_ratio,user_id,order_number,avg_hhi
0,2,0.666667,0.333333,8.200920,9,1.0,202279,3,0.135802
1,3,1.000000,0.625000,9.754306,8,1.0,205970,16,0.187500
2,4,0.846154,0.076923,7.812530,13,1.0,178520,36,0.159763
3,5,0.692308,0.230769,8.053129,26,1.0,156122,42,0.071006
4,6,0.000000,0.000000,4.629503,3,1.0,22352,4,0.333333


## 4) Trend Curves by Order Number (Section 4.1)

In [6]:
trend_by_order_number = (
    order_metrics.groupby("order_number", sort=True)
    .agg(
        avg_popular_share=("avg_popular_share", "mean"),
        avg_unique_ratio=("avg_unique_ratio", "mean"),
        avg_items=("avg_items", "mean"),
        n_user_orders=("order_id", "size"),
    )
    .reset_index()
)

trend_extra_metrics = (
    order_metrics.groupby("order_number", sort=True)
    .agg(
        avg_mean_log_popularity=("avg_mean_log_popularity", "mean"),
        avg_ultra_popular_share=("avg_ultra_popular_share", "mean"),
        n_user_orders=("order_id", "size"),
    )
    .reset_index()
)

hhi_trend = (
    order_metrics.groupby("order_number", sort=True)
    .agg(avg_hhi=("avg_hhi", "mean"))
    .reset_index()
)

trend_by_order_number.head(20)

,order_number,avg_popular_share,avg_unique_ratio,avg_items,n_user_orders
0,1,0.792284,1.0,10.077484,206209
1,2,0.793216,1.0,9.933281,206209
2,3,0.794167,1.0,9.944915,206209
3,4,0.796408,1.0,9.989398,182223
4,5,0.799270,1.0,10.012796,162633
5,6,0.800851,1.0,10.051602,146468
6,7,0.802542,1.0,10.057813,132618
7,8,0.804393,1.0,10.082436,120918
8,9,0.806052,1.0,10.119103,110728
9,10,0.807636,1.0,10.115481,101696


In [7]:
trend_first20 = trend_by_order_number.loc[trend_by_order_number["order_number"] <= 20].copy()
trend_extra_first20 = trend_extra_metrics.loc[trend_extra_metrics["order_number"] <= 20].copy()
hhi_first20 = hhi_trend.loc[hhi_trend["order_number"] <= 20].copy()

trend_first20

,order_number,avg_popular_share,avg_unique_ratio,avg_items,n_user_orders
0,1,0.792284,1.0,10.077484,206209
1,2,0.793216,1.0,9.933281,206209
2,3,0.794167,1.0,9.944915,206209
3,4,0.796408,1.0,9.989398,182223
4,5,0.799270,1.0,10.012796,162633
5,6,0.800851,1.0,10.051602,146468
6,7,0.802542,1.0,10.057813,132618
7,8,0.804393,1.0,10.082436,120918
8,9,0.806052,1.0,10.119103,110728
9,10,0.807636,1.0,10.115481,101696


## 5) Slopes and Key Table 1 Numbers (Section 4.1)

In [8]:
def slope_over_orders(df: pd.DataFrame, y_col: str) -> float:
    x = df["order_number"].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    return float(np.polyfit(x, y, 1)[0])

key_table1 = pd.DataFrame({
    "metric": [
        "popular-item share",
        "ultra-popular share",
        "mean log global popularity",
        "within-basket HHI",
    ],
    "order_1": [
        float(trend_first20.loc[trend_first20["order_number"] == 1, "avg_popular_share"].iloc[0]),
        float(trend_extra_first20.loc[trend_extra_first20["order_number"] == 1, "avg_ultra_popular_share"].iloc[0]),
        float(trend_extra_first20.loc[trend_extra_first20["order_number"] == 1, "avg_mean_log_popularity"].iloc[0]),
        float(hhi_first20.loc[hhi_first20["order_number"] == 1, "avg_hhi"].iloc[0]),
    ],
    "order_20": [
        float(trend_first20.loc[trend_first20["order_number"] == 20, "avg_popular_share"].iloc[0]),
        float(trend_extra_first20.loc[trend_extra_first20["order_number"] == 20, "avg_ultra_popular_share"].iloc[0]),
        float(trend_extra_first20.loc[trend_extra_first20["order_number"] == 20, "avg_mean_log_popularity"].iloc[0]),
        float(hhi_first20.loc[hhi_first20["order_number"] == 20, "avg_hhi"].iloc[0]),
    ],
    "slope_per_order": [
        slope_over_orders(trend_first20, "avg_popular_share"),
        slope_over_orders(trend_extra_first20, "avg_ultra_popular_share"),
        slope_over_orders(trend_extra_first20, "avg_mean_log_popularity"),
        slope_over_orders(hhi_first20, "avg_hhi"),
    ],
})
key_table1

,metric,order_1,order_20,slope_per_order
0,popular-item share,0.792284,0.819243,0.001463
1,ultra-popular share,0.399913,0.435170,0.002042
2,mean log global popularity,8.615294,8.808616,0.010773
3,within-basket HHI,0.269341,0.262720,-0.000574


## 6) First-vs-Last User Analysis (Section 3.4 and 4.2)

In [9]:
user_first = (
    orders_prior.sort_values(["user_id", "order_number"])
    .groupby("user_id", as_index=False)
    .first()[["user_id", "order_id"]]
    .rename(columns={"order_id": "first_order_id"})
)
user_last = (
    orders_prior.sort_values(["user_id", "order_number"])
    .groupby("user_id", as_index=False)
    .last()[["user_id", "order_id"]]
    .rename(columns={"order_id": "last_order_id"})
)
user_first_last = user_first.merge(user_last, on="user_id", how="inner", validate="one_to_one")

order_reduced = order_metrics[["order_id", "avg_popular_share", "avg_unique_ratio"]].copy()
user_first_last = user_first_last.merge(
    order_reduced.add_prefix("first_"),
    left_on="first_order_id",
    right_on="first_order_id",
    how="left",
    validate="one_to_one",
)
user_first_last = user_first_last.merge(
    order_reduced.add_prefix("last_"),
    left_on="last_order_id",
    right_on="last_order_id",
    how="left",
    validate="one_to_one",
)

user_first_last["delta_popular_share"] = user_first_last["last_avg_popular_share"] - user_first_last["first_avg_popular_share"]
user_first_last["delta_unique_ratio"] = user_first_last["last_avg_unique_ratio"] - user_first_last["first_avg_unique_ratio"]

first_vs_last_summary = pd.DataFrame([
    {
        "metric": "popular_share",
        "mean_first": user_first_last["first_avg_popular_share"].mean(),
        "mean_last": user_first_last["last_avg_popular_share"].mean(),
        "mean_change_last_minus_first": user_first_last["delta_popular_share"].mean(),
        "median_change_last_minus_first": user_first_last["delta_popular_share"].median(),
    },
    {
        "metric": "unique_ratio",
        "mean_first": user_first_last["first_avg_unique_ratio"].mean(),
        "mean_last": user_first_last["last_avg_unique_ratio"].mean(),
        "mean_change_last_minus_first": user_first_last["delta_unique_ratio"].mean(),
        "median_change_last_minus_first": user_first_last["delta_unique_ratio"].median(),
    },
])

user_delta_quantiles = pd.DataFrame({
    "delta_popular_share": user_first_last["delta_popular_share"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]),
    "delta_unique_ratio": user_first_last["delta_unique_ratio"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]),
})

first_vs_last_summary

,metric,mean_first,mean_last,mean_change_last_minus_first,median_change_last_minus_first
0,popular_share,0.792284,0.790798,-0.001485,0.0
1,unique_ratio,1.000000,1.000000,0.000000,0.0


In [10]:
user_delta_quantiles

,delta_popular_share,delta_unique_ratio
0.10,-0.265455,0.0
0.25,-0.111111,0.0
0.50,0.000000,0.0
0.75,0.111111,0.0
0.90,0.260870,0.0


## 7) Export Tables (same filenames as paper outputs)

In [11]:
trend_by_order_number.to_csv(OUT_DIR / "trend_by_order_number.csv", index=False)
trend_first20.to_csv(OUT_DIR / "trend_first20.csv", index=False)
trend_extra_metrics.to_csv(OUT_DIR / "trend_extra_metrics.csv", index=False)
trend_extra_first20.to_csv(OUT_DIR / "trend_extra_first20.csv", index=False)
hhi_trend.to_csv(OUT_DIR / "hhi_trend_by_order_number.csv", index=False)
hhi_first20.to_csv(OUT_DIR / "hhi_first20.csv", index=False)
first_vs_last_summary.to_csv(OUT_DIR / "first_vs_last_summary.csv", index=False)
user_delta_quantiles.to_csv(OUT_DIR / "user_delta_quantiles.csv")

key_numbers = pd.DataFrame(
    {
        "value": [
            float(orders_prior["user_id"].nunique()),
            float(len(orders_prior)),
            q90,
            user_first_last["delta_popular_share"].mean(),
            user_first_last["delta_popular_share"].median(),
            user_first_last["delta_unique_ratio"].mean(),
            user_first_last["delta_unique_ratio"].median(),
            slope_over_orders(trend_first20, "avg_popular_share"),
            slope_over_orders(trend_first20, "avg_unique_ratio"),
            slope_over_orders(hhi_first20, "avg_hhi"),
        ]
    },
    index=[
        "n_users",
        "n_prior_orders",
        "popular_threshold_count_q90",
        "mean_delta_popular_share",
        "median_delta_popular_share",
        "mean_delta_unique_ratio",
        "median_delta_unique_ratio",
        "first20_popular_share_slope_per_order",
        "first20_unique_ratio_slope_per_order",
        "first20_hhi_slope_per_order",
    ],
)
key_numbers.to_csv(OUT_DIR / "key_numbers.csv")

key_numbers_extra = pd.DataFrame(
    {
        "value": [
            q99,
            slope_over_orders(trend_extra_first20, "avg_mean_log_popularity"),
            slope_over_orders(trend_extra_first20, "avg_ultra_popular_share"),
            float(trend_extra_first20.loc[trend_extra_first20["order_number"] == 1, "avg_ultra_popular_share"].iloc[0]),
            float(trend_extra_first20.loc[trend_extra_first20["order_number"] == 20, "avg_ultra_popular_share"].iloc[0]),
        ]
    },
    index=[
        "ultra_popular_q99_threshold_count",
        "first20_log_popularity_slope_per_order",
        "first20_ultra_popular_share_slope_per_order",
        "order1_avg_ultra_popular_share",
        "order20_avg_ultra_popular_share",
    ],
)
key_numbers_extra.to_csv(OUT_DIR / "key_numbers_extra.csv")

print("Wrote outputs to:", OUT_DIR)

Wrote outputs to: /Users/cameronwang/Desktop/Foundations of Algorithmic Unfairness/unfairness-instacart-project/analysis/final_presentation_analysis_outputs
